# NASA C-MAPSS Predictive Maintenance Pipeline — Final Summary

## Project Overview

This notebook provides a comprehensive, standalone summary of the entire **NASA C-MAPSS Predictive Maintenance** project. The goal of this project is to predict the **Remaining Useful Life (RUL)** of turbofan jet engines using machine-learning and deep-learning techniques applied to the Commercial Modular Aero-Propulsion System Simulation (C-MAPSS) dataset.

---

### Table of Contents
1. [Dataset Description](#1)
2. [Exploratory Data Analysis (EDA)](#2)
3. [Feature Engineering & Data Preparation](#3)
4. [Classical Machine Learning Models](#4)
5. [Extended ML Tasks (Classification, SMOTE, PCA)](#5)
6. [Deep Learning — LSTM Models](#6)
7. [Model Comparison & Final Results](#7)
8. [Key Findings & Recommendations](#8)


<a id="1"></a>
## 1. Dataset Description

The **C-MAPSS FD001** dataset is one of four subsets of the NASA Turbofan Engine Degradation Simulation dataset. It simulates run-to-failure trajectories for a fleet of turbofan engines under a single operating condition and a single fault mode.

| Property | Value |
|---|---|
| **Training engines** | 100 |
| **Test engines** | 100 |
| **Operating conditions** | 1 |
| **Fault modes** | 1 (HPC degradation) |
| **Sensors per engine** | 21 |
| **Operational settings** | 3 |

Each row in the training set contains:
- `unit` — Engine ID
- `cycle` — Time step (operational cycle)
- `op1`, `op2`, `op3` — Three operational setting channels
- `s1` through `s21` — 21 sensor measurement channels

In the training set, each engine runs until failure. The test set is truncated at some point before failure, and a separate *RUL_FD001.txt* file provides the ground-truth remaining cycles for each test engine.


<a id="2"></a>
## 2. Exploratory Data Analysis (EDA)

**Notebook:** `01_EDA.ipynb`

### Objectives
- Understand the structure and statistical properties of the raw sensor data.
- Identify constant or near-constant sensors that carry no predictive value.
- Visualize engine degradation patterns over time.

### Key Findings
1. **Flat sensors:** Several sensors (`s1`, `s5`, `s6`, `s10`, `s16`, `s18`, `s19`) show zero or near-zero variance across all engines and cycles. These were dropped during feature engineering as they contribute only noise.
2. **Operational settings:** The three operational settings (`op1`, `op2`, `op3`) also exhibit very low variance in FD001 (single operating condition), so they were excluded from the feature set.
3. **Degradation trends:** Sensors such as `s2`, `s3`, `s4`, `s7`, `s8`, `s9`, `s11`, `s12`, `s13`, `s14`, `s15`, `s17`, `s20`, `s21` show clear degradation trends — monotonically increasing or decreasing as each engine approaches failure.
4. **Engine lifetimes:** Engine lifetimes range widely (from ~128 to ~362 cycles), providing diverse degradation trajectories for model training.


<a id="3"></a>
## 3. Feature Engineering & Data Preparation

**Notebook:** `02_Feature_Engineering.ipynb`

### RUL Label Construction
- The RUL for each cycle is computed as `max_cycle_for_engine - current_cycle`.
- A **cap of 125 cycles** is applied: RUL values above 125 are clipped to 125. This reflects the practical insight that engines in the early phase of their life are effectively "as good as new" and need not be distinguished.

### Feature Selection
After EDA, the following features were **dropped**:
- Constant/near-constant sensors: `s1`, `s5`, `s6`, `s10`, `s16`, `s18`, `s19`
- Operational settings: `op1`, `op2`, `op3`

The remaining **14 sensor channels** were retained.

### Rolling-Window Features
To capture temporal dynamics (trends, smoothing), rolling-window statistics were computed for each retained sensor with a **window size of 30 cycles**:
- **Rolling mean** — captures smoothed sensor trends
- **Rolling standard deviation** — captures variability / signal noise

This expanded the feature set from 14 raw sensor columns to **14 raw + 14 mean + 14 std = 42 features** per time step.

### Normalization
All features were scaled to [0, 1] using `MinMaxScaler`, fitted on the training set and applied consistently to the test set.

### Binary Classification Label
A binary label `will_fail_30` was created for classification tasks:
- `1` if RUL ≤ 30 (engine approaching imminent failure)
- `0` otherwise

### Outputs
- `processed_train.csv` — Training data with all features, RUL, and binary labels
- `processed_test.csv` — Test data with all features (no RUL column)
- `RUL_FD001.txt` — Ground-truth RUL for each test engine


<a id="4"></a>
## 4. Classical Machine Learning Models

**Notebook:** `03_Classical_ML.ipynb`

### Models & Hyperparameter Tuning
Two tree-based ensemble methods were trained for the **regression** task (predicting continuous RUL):

| Model | Tuning Method | Key Hyperparameters |
|---|---|---|
| **Random Forest** | `GridSearchCV` (3-fold CV) | `n_estimators`, `max_depth`, `min_samples_split` |
| **XGBoost** | `GridSearchCV` (3-fold CV) | `n_estimators`, `max_depth`, `learning_rate`, `subsample` |

### Evaluation Metrics
All models were evaluated on the **test set** (100 engines) using three metrics:

| Metric | Description |
|---|---|
| **RMSE** | Root Mean Squared Error — penalizes large errors |
| **MAE** | Mean Absolute Error — average magnitude of errors |
| **NASA Score** | Asymmetric scoring function from the C-MAPSS competition: penalizes late predictions (failure occurs before predicted) more heavily than early predictions. Formula: `Σ exp(d/10)−1` if `d≥0`, `Σ exp(−d/13)−1` if `d<0`, where `d = predicted − actual`. |

### Classical ML Results (FD001 Test Set)

| Model | RMSE | MAE | NASA Score |
|---|---|---|---|
| Random Forest | 18.30 | 13.70 | 734.52 |
| XGBoost | 17.11 | 12.84 | 586.42 |

XGBoost outperformed Random Forest across all three metrics.


<a id="5"></a>
## 5. Extended ML Tasks

**Notebook:** `03B_Extended_ML.ipynb`

### Additional Regression Models

| Model | RMSE | MAE | NASA Score | Notes |
|---|---|---|---|---|
| **KNN Regressor** | 17.58 | 13.56 | 662.57 | Best `n_neighbors` selected via GridSearchCV |
| **SVR** | 16.70 | 12.38 | 581.83 | Trained on 5,000 sample subset (for scalability); RBF and linear kernels tested |

SVR achieved the best NASA Score among all classical ML models.

### Binary Classification (Failure within 30 cycles)
Four classifiers were trained to predict the binary label `will_fail_30`:

| Classifier | Accuracy | Precision | Recall | F1 |
|---|---|---|---|---|
| Random Forest Classifier | — | — | — | — |
| XGBoost Classifier | — | — | — | — |
| KNN Classifier | — | — | — | — |
| SVC | — | — | — | — |

*Results were collected in the notebook's metric aggregation table; confusion matrices were plotted for each classifier.*

### SMOTE (Handling Imbalanced Data)
To simulate a realistic imbalanced scenario (10% minority class), the training data was downsampled and then rebalanced using **SMOTE (Synthetic Minority Over-sampling Technique)**.

| Condition | F1 Score |
|---|---|
| Imbalanced baseline (RF) | Lower |
| After SMOTE (RF) | Improved |

This demonstrated that SMOTE effectively improves minority-class recall at a manageable cost to precision.

### Maintenance Thresholding
Using the top-3 most important features (identified by Random Forest feature importances), a brute-force threshold search was conducted to find interpretable rules of the form:

> "If *Feature_A ≥ threshold_1* AND *Feature_B ≤ threshold_2* AND *Feature_C ≥ threshold_3* → X% failure probability"

Rules exceeding 50% (and especially 85%) failure probability were flagged as **HIGH RISK** maintenance triggers.

### PCA Visualization
PCA was applied to the test features to produce 2D scatter plots:
- **Ground-truth failure states** (Healthy vs. Failing) showed visual clustering in principal component space.
- **K-Means clustering** (k=3) on the same PCA projection revealed natural groupings that partially aligned with the failure states.


<a id="6"></a>
## 6. Deep Learning — LSTM Models

**Notebook:** `04_Deep_Learning_LSTM.ipynb`

### Motivation
Classical ML models treat each time step independently. LSTMs (Long Short-Term Memory networks) are designed to model **sequential / temporal dependencies**, making them well-suited for sensor time-series data where degradation evolves over consecutive cycles.

### Data Reshaping — Sliding Windows
The processed feature data was reshaped into **sliding windows of 30 time steps**:
- For each engine, windows of shape `(30, 42)` (30 time steps × 42 features) were generated.
- The target RUL for each window is the RUL at the last time step in the window.
- For the test set, only the **last window** per engine was used for prediction.
- Training shapes: `X_train (17631, 30, 42)`, `y_train (17631,)`
- Test shapes: `X_test (100, 30, 42)`

### Architectures

| Architecture | Hidden Size | Layers | Dropout | Special |
|---|---|---|---|---|
| **Baseline LSTM** | 100 | 2 | 0.2 | Standard LSTM → FC |
| **Deep LSTM** | 128 | 3 | 0.3 | LSTM → BatchNorm → FC |

Both models use only the **last time step's output** from the LSTM to produce a single RUL prediction via a fully connected layer.

### Training Strategy
- **Optimizer:** Adam (lr=0.001 initial, then 0.0005 for extended training)
- **Loss:** MSE
- **Batch size:** 1024
- **Train/Val split:** 80/20 random split
- **Learning rate scheduler:** `ReduceLROnPlateau` (factor=0.5, patience=5–7)
- **Early stopping:** patience=10–15 epochs
- **Checkpointing:** Best model weights saved based on validation loss
- **Device:** GPU (CUDA) when available

### Training Progression

**Phase 1 (50 epochs each):**
- Baseline LSTM achieved best val loss: **2159.30** (no early stop)
- Deep LSTM achieved best val loss: **3876.95** (early stopped at epoch 30)
- ✅ **Baseline LSTM selected** for extended training.

**Phase 2 — Extended Training (100 more epochs with lower LR):**
- Continued from the Phase 1 checkpoint with lr=0.0005
- Final best val loss: **157.48** — a dramatic improvement

### LSTM Results (FD001 Test Set)

| Model | RMSE | MAE | NASA Score |
|---|---|---|---|
| LSTM (Extended) | **16.29** | **11.79** | **436.74** |

The extended LSTM model achieved the **best performance across all metrics**, significantly outperforming all classical ML approaches.


<a id="7"></a>
## 7. Model Comparison & Final Results

**Notebook:** `05_Model_Comparison.ipynb`

### Unified Regression Results (FD001 Test Set — Lower is Better)

| Model | RMSE | MAE | NASA Score |
|---|---|---|---|
| Random Forest | 18.30 | 13.70 | 734.52 |
| XGBoost | 17.11 | 12.84 | 586.42 |
| KNN Regressor | 17.58 | 13.56 | 662.57 |
| SVR | 16.70 | 12.38 | 581.83 |
| **LSTM (Extended)** | **16.29** | **11.79** | **436.74** |

### Rankings

| Rank | By RMSE | By MAE | By NASA Score |
|---|---|---|---|
| 1 | LSTM (16.29) | LSTM (11.79) | LSTM (436.74) |
| 2 | SVR (16.70) | SVR (12.38) | SVR (581.83) |
| 3 | XGBoost (17.11) | XGBoost (12.84) | XGBoost (586.42) |
| 4 | KNN (17.58) | KNN (13.56) | KNN (662.57) |
| 5 | Random Forest (18.30) | Random Forest (13.70) | Random Forest (734.52) |

### Key Observations
- The **LSTM model is the clear winner** across all three metrics, confirming the value of modeling temporal sensor dynamics.
- Among classical models, **SVR** edges out XGBoost, particularly on the NASA Score (the most operationally relevant metric).
- **Random Forest** consistently ranked last, suggesting that individual decision trees struggle with the smooth degradation curves.
- The **NASA Score** metric reveals the biggest gaps between models because it exponentially penalizes dangerous late predictions.


<a id="8"></a>
## 8. Key Findings & Recommendations

### Achievements
1. **End-to-end pipeline:** A complete workflow from raw data ingestion through EDA, feature engineering, model training, and evaluation.
2. **Robust feature engineering:** Rolling-window statistics (mean, std over 30 cycles) effectively captured temporal degradation patterns.
3. **Multiple model comparison:** Five regression models and four classifiers were trained and rigorously compared.
4. **Domain-aware evaluation:** The asymmetric NASA scoring function was implemented as a custom utility (`utils/nasa_score.py`) and used consistently across all models.
5. **Deep learning advantage:** The LSTM model demonstrated a clear superiority for this sequential prediction task, achieving a ~25% reduction in NASA Score vs. the best classical model.
6. **Practical maintenance rules:** Threshold-based rules were derived from feature importances, providing interpretable guidelines for maintenance scheduling.

### Limitations
- Only the **FD001 subset** (single operating condition, single fault mode) was used. Results may differ on FD002–FD004.
- SVR training was limited to a **5,000-sample subset** due to computational cost; full-data SVR could yield different results.
- LSTM hyperparameters (hidden size, layers, dropout) were explored with only two configurations; a broader search may improve results.

### Recommendations for Future Work
1. **Extended datasets:** Evaluate on FD002, FD003, and FD004 (multiple operating conditions and fault modes) to assess model robustness.
2. **Architecture exploration:** Try Bi-directional LSTMs, GRUs, Temporal Convolutional Networks (TCNs), or Transformer-based architectures.
3. **Deeper hyperparameter search:** Use tools like Optuna or Ray Tune for systematic LSTM hyperparameter optimization.
4. **Production readiness:** Convert notebook-based pipelines into modular Python scripts for deployment (see `simplify_dl.py`, `simplify_nb.py`).
5. **Uncertainty quantification:** Implement Monte Carlo Dropout or ensemble methods to provide confidence intervals on RUL predictions.
6. **Online learning:** Explore incremental / transfer learning so models can adapt to new engines without full retraining.


---

## Project File Structure

```
NASA CMaps/
├── data/
│   ├── train_FD001.txt          # Raw training data
│   ├── test_FD001.txt           # Raw test data
│   ├── RUL_FD001.txt            # Ground-truth test RUL
│   ├── processed_train.csv      # Engineered training features
│   ├── processed_test.csv       # Engineered test features
│   ├── classical_metrics.csv    # RF & XGBoost metrics
│   ├── lstm_metrics.csv         # LSTM metrics
│   └── extended_ml_metrics.csv  # KNN, SVR, Classifier metrics
├── notebooks/
│   ├── 01_EDA.ipynb                   # Exploratory Data Analysis
│   ├── 02_Feature_Engineering.ipynb   # Feature engineering pipeline
│   ├── 03_Classical_ML.ipynb          # RF & XGBoost regression
│   ├── 03B_Extended_ML.ipynb          # KNN, SVR, Classification, SMOTE, PCA
│   ├── 04_Deep_Learning_LSTM.ipynb    # LSTM training & evaluation
│   ├── 05_Model_Comparison.ipynb      # Unified comparison & visualizations
│   └── 06_Final_Summary.ipynb         # ← This notebook
├── utils/
│   └── nasa_score.py            # Custom NASA scoring function
└── ...
```

---

## NASA Scoring Function (Reference)

The domain-specific scoring metric used throughout this project:

```python
import numpy as np

def nasa_score(y_true, y_pred):
    d = y_pred - y_true
    score = np.sum(np.where(d < 0, np.exp(-d/13) - 1, np.exp(d/10) - 1))
    return score
```

This function is **asymmetric by design**:
- Late predictions (`d > 0`, engine fails before predicted) are penalized with `exp(d/10)` — a steeper exponential — because in real-world operations, a missed failure is far more costly than premature maintenance.
- Early predictions (`d < 0`) use the gentler `exp(-d/13)` scaling.

---

*End of Final Summary*
